# Integrated Dietary Network Analysis (FIXED)
## GGM Analysis와 Co-occurrence/Centrality Analysis의 통합

**수정 사항:**
- ✅ 올바른 식품군 데이터 선택 (40-44, 45-47, 50-53번 열)
- ✅ 개선된 co-occurrence 함수 (70th percentile 임계값)
- ✅ 수정된 variable_mapping

### 분석 목표
- GGM에서 발견된 커뮤니티와 Co-occurrence 패턴 비교
- 두 방법론에서 일관되게 나타나는 핵심 식품군 식별
- MetS와 관련된 식습관 패턴의 통합적 이해

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import GraphicalLassoCV
from scipy import stats
from scipy.stats import spearmanr

plt.rcParams['font.family'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 10)

## 1. 데이터 로딩 및 전처리 (FIXED)

In [ ]:
# Load data
data = pd.read_csv("../db/processed_data/total_only_org.csv")

# Extract food groups (detailed 19 variables for GGM)
food_groups_detailed = data.iloc[:, 35:54]  # Columns 35-53
food_names_detailed = list(food_groups_detailed.columns)

# FIXED: Extract aggregated 12 food groups (for co-occurrence)
# 올바른 식품군: 40-44, 45-47, 50-53번 열
aggregated_food_cols = [
    'Grain Products', 'Protein Foods', 'Vegetables', 'Dairy Products',
    'Fruits', 'Fried Foods', 'High Fat Meat', 'Processed Foods',
    'Sugar-Sweetened Beverages', 'Additional Salt Use',
    'Salty Food Consumption', 'Sweet Food Consumption'
]
food_groups_agg = data[aggregated_food_cols]
food_names_agg = list(food_groups_agg.columns)

# Extract health variables
health_vars = data[['BMI(kg/m2)', 'Systolic blood pressure (mmHg)',
                    'Diastolic Blood Pressure (mmHg)', 'Total Cholesterol (mg/dL)',
                    'Triglycerides (mg/dL)', 'LDL-C (mg/dL)', 'HDL-C (mg/dL)',
                    'Fasting glucose (mg/dL)']]
health_vars.columns = ['BMI', 'SBP', 'DBP', 'TC', 'TG', 'LDL', 'HDL', 'FG']

# Extract MetS components
mets_components = data[['Increased waist circumference', 'Elevated blood pressure',
                        'Impaired fasting glucose', 'Elevated triglycerides',
                        'Decreased HDL-C', 'MetS']]
mets_components.columns = ['WC', 'U_BP', 'IFG', 'U_TG', 'D_HDL', 'MetS']

# Combine data
combined_data = pd.concat([food_groups_detailed, food_groups_agg, health_vars, mets_components], axis=1)
combined_data = combined_data.dropna()

print(f"Total sample size: {len(combined_data):,}")
print(f"Detailed food variables (GGM): {len(food_names_detailed)}")
print(f"Aggregated food groups (Co-occurrence): {len(food_names_agg)}")
print(f"\nFood names (aggregated): {food_names_agg}")
print(f"\nMetS prevalence: {combined_data['MetS'].sum()} ({combined_data['MetS'].mean()*100:.1f}%)")